# Regression and variance-partition analyses

This notebook reproduces the regression results of the paper: how much of the variation in AF3's accuracy
(RNA TM-score, complex TM-score, interface DockQ) the six groups of features explain, and how the explained
variance is shared between them (Shapley partition). It runs the same pipeline as `regression_analysis/reproduce_all.sh`
and then shows the main tables and Figures 1 and 2.

Running everything takes roughly one to two hours on a laptop. All cross-validation uses a fixed seed, so a rerun
gives the same numbers.

Before the first run, install the environment and download the data (see the README):

```
conda env create -f environment.yml
conda activate af3_eval
bash fetch_data.sh
```

In [ ]:
import glob, os, subprocess, sys
import pandas as pd
from IPython.display import Image, display

REPO = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
RA = os.path.join(REPO, "regression_analysis")
needed = os.path.join(RA, "data", "post_pre", "a2021_b2021_protein_rna", "a2021.db")
if not os.path.exists(needed):
    print("Databases not found, downloading them from Zenodo ...")
    subprocess.run([sys.executable, os.path.join(REPO, "utils", "fetch_zenodo.py")], check=True)
print("repository:", REPO)

## Run the analysis (one command)

Preprocessing of the databases, the grouped cross-validated ridge regressions, the Shapley partition and the
stability selection. The output of the pipeline is printed below as it runs.

In [ ]:
proc = subprocess.Popen(["bash", "reproduce_all.sh"], cwd=RA, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
print("finished with exit code", proc.wait())

## Explained variance and its partition

One table per analysed set and target: the cross-validated R² of all feature groups together and the share of each
group (G0 AF3 confidence, G1 evolution, G2 complexity, G3 RNA-intrinsic, G4 predicted interface, G5 experimental
context).

In [ ]:
for path in sorted(glob.glob(os.path.join(RA, "variance_partitioning_final", "**", "shapley_r2.csv"), recursive=True)):
    print(os.path.relpath(os.path.dirname(path), RA))
    display(pd.read_csv(path))

## Figures 1 and 2

In [ ]:
for name in ("Figure1", "Figure2"):
    hits = sorted(glob.glob(os.path.join(RA, "publication_figures", "combined", f"{name}.png")))
    if hits:
        display(Image(filename=hits[0]))
    else:
        print(f"{name}.png not found; see the output of the pipeline above")